In [31]:
import mlflow
import pandas as pd
import numpy as np
import optuna
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [32]:
# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_experiment("Hyperparameter Wine Quality Tuning Experiment")

<Experiment: artifact_location='file:///c:/Users/jeroen.vander.putten/Development/Learning/Python/github_portfolio_jeroen/project_MLflow/mlruns/4', creation_time=1768736091216, experiment_id='4', last_update_time=1768736091216, lifecycle_stage='active', name='Hyperparameter Wine Quality Tuning Experiment', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [33]:
# Load and preprocess the data
data = pd.read_csv("WineQt.csv")

# Rename columns to replace spaces with underscores
data.rename(columns=lambda x: x.replace(' ', '_'), inplace=True)

# Create a binary target variable for high quality wines
high_quality = (data.quality >= 7).astype(int)
data['high_quality'] = high_quality

#
X = data.drop(["quality", "high_quality", "Id"], axis=1)
y = data["high_quality"]

# Split out the data
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=123)


In [34]:
def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:

        # Hyperparameters and search space
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 32),
            "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=10),
            "max_features": trial.suggest_float("max_features", 0.2, 0.8),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 10),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 256),
            "bootstrap": True,
            "n_jobs": -1,
            "random_state": 42,
        }

        # Log current trial's parameters
        mlflow.log_params(params)
        mlflow.log_param("trial_number", trial.number) # Log trial number
        mlflow.log_param("run_id", child_run.info.run_id) # Log run ID

        # Train and evaluate the model
        rf = sklearn.ensemble.RandomForestClassifier(**params)  # Instantiate the model
        rf.fit(X_train, y_train)  # Fit to training data
        y_score = rf.predict_proba(X_test)[:, 1]  # Get predicted probabilities
        roc_auc = sklearn.metrics.roc_auc_score(y_test, y_score)  # Calculate ROC AUC

        # Log current trial's metric
        mlflow.log_metric("roc_auc_score", roc_auc)

        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)

        return roc_auc

In [35]:

def champion_callback(study, frozen_trial):
    """
    Logging callback that will report when a new trial iteration improves upon existing
    best trial values.
    """
    # Keep the "winner" stored in study attrs
    winner = study.user_attrs.get("winner", None)

    # Only act when this trial is the best so far
    if study.best_trial.number == frozen_trial.number:
        study.set_user_attr("winner", study.best_value)

        if winner is not None:
            # % improvement vs previous winner (guard against divide by zero)
            denom = abs(winner) if abs(winner) > 0 else 1e-12
            improvement_percent = (abs(study.best_value - winner) / denom) * 100
            print(
                f"Trial {frozen_trial.number} achieved value: {frozen_trial.value} with "
                f"{improvement_percent:.4f}% improvement"
            )
        else:
            print(f"Initial trial {frozen_trial.number} achieved value: {frozen_trial.value}")


In [ ]:
optuna.logging.set_verbosity(optuna.logging.ERROR) ## Suppress optuna info logs

# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="Random Forest") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(seed=123)
    )
    study.optimize(objective, n_trials=n_trials, callbacks=[champion_callback])

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metric("best_roc_auc_score", study.best_value)
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)
        mlflow.log_param("best_trial_number", study.best_trial.number)

    # Log a fit model instance (FIX: **best_params)
    rf = sklearn.ensemble.RandomForestClassifier(
        **study.best_params,   # use best hyperparameters found
        n_jobs=-1,             # keep deterministic/speed even if not in best_params
        random_state=42 
    )
    rf.fit(X_train, y_train) # Fit to training data with best hyperparameters

    # Log the final best model (not every individual trials)
    mlflow.sklearn.log_model(rf, artifact_path="random_forest_model") 

Initial trial 0 achieved value: 0.8675742574257426
Trial 4 achieved value: 0.8677392739273928 with 0.0190% improvement
Trial 5 achieved value: 0.8757425742574256 with 0.9223% improvement
Trial 20 achieved value: 0.875990099009901 with 0.0283% improvement


2026/01/18 14:46:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/18 14:46:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
